# Data Ingestion — MySQL (Aiven) + Qdrant (JobPath)

Notebook ini membersihkan `dataset/jobs.jsonl` lalu memuatnya ke dua database:
- **MySQL (Aiven)** — tabel relasional `jobs` untuk SQL Agent (Smart Search, Market Insight).
- **Qdrant** — collection vektor `jobs` untuk RAG (Job Chat, CV Matcher).

**Idempotency & resume:**
- `job_id` = UUID5 deterministik dari isi lowongan → stabil antar-run, jadi jembatan MySQL↔Qdrant.
- MySQL memakai `INSERT ... ON DUPLICATE KEY UPDATE` (run ulang tidak menggandakan).
- Qdrant memakai `point_id = job_id` + `upsert` (id sama menimpa, bukan menambah).
- **Resume**: sebelum embedding, daftar `job_id` yang sudah ada di Qdrant diambil; hanya sisanya yang diproses. Jika gagal di tengah, run berikutnya lanjut dari yang belum masuk — biaya embedding tidak terbuang.

**Konfigurasi** dibaca dari `.env` (lihat `.env.example`).

## Bagian 1 — Setup & koneksi

In [4]:
import os
import re
import json
import uuid
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from dotenv import load_dotenv, find_dotenv

# Kernel Jupyter berjalan dgn cwd = folder notebook (notebooks/). .env di root repo
# dicari ke atas dari cwd; ROOT dipakai untuk meresolusi path relatif lain (mis. CA cert).
dotenv_path = find_dotenv(usecwd=True)
load_dotenv(dotenv_path, override=True)
ROOT = Path(dotenv_path).resolve().parent if dotenv_path else Path.cwd()


def resolve_path(p):
    """Resolusikan path relatif terhadap root repo (bukan cwd notebook)."""
    p = Path(p)
    return p if p.is_absolute() else (ROOT / p)


EMB_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
QDRANT_COLLECTION = os.getenv("QDRANT_COLLECTION", "jobs")
BATCH_SIZE = int(os.getenv("INGEST_BATCH_SIZE", "32"))
VECTOR_SIZE = 1536  # text-embedding-3-small

print("Root repo       :", ROOT)
print("Embedding model :", EMB_MODEL)
print("Qdrant collection:", QDRANT_COLLECTION)
print("Batch size      :", BATCH_SIZE)

Root repo       : D:\codes\Finpro-JobApp
Embedding model : text-embedding-3-small
Qdrant collection: jobs
Batch size      : 32


In [12]:
# Klien layanan
from openai import OpenAI
from qdrant_client import QdrantClient
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

client_oai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# timeout dinaikkan agar upsert batch ke Qdrant Cloud tidak putus di jaringan lambat
qc = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"), timeout=120)

mysql_url = URL.create(
    "mysql+pymysql",
    username=os.getenv("MYSQL_USER"),
    password=os.getenv("MYSQL_PASSWORD"),
    host=os.getenv("MYSQL_HOST"),
    port=int(os.getenv("MYSQL_PORT")),
    database=os.getenv("MYSQL_DATABASE"),
)
ca_path = str(resolve_path(os.getenv("MYSQL_SSL_CA")))  # Aiven wajib SSL
engine = create_engine(mysql_url, connect_args={"ssl": {"ca": ca_path}})

# Uji koneksi cepat
with engine.connect() as c:
    print("MySQL  :", c.execute(text("SELECT VERSION()")).scalar())
print("Qdrant :", [c.name for c in qc.get_collections().collections])

MySQL  : 8.4.8
Qdrant : ['jobs']


## Bagian 2 — Muat & bersihkan data

Menerapkan seluruh keputusan dari `data_exploration.ipynb`: dedup, parse salary, pecah lokasi + `work_arrangement`, normalisasi `work_type`, anonim → NULL, turunkan `seniority_level`, dan generate `job_id`. Fungsi feature untuk model gaji identik dengan `salary_ml.ipynb` agar konsisten.

In [3]:
# --- Fungsi transformasi (konsisten dgn exploration & salary_ml) ---
SENIORITY_KEYWORDS = ["intern", "junior", "senior", "lead", "principal", "manager", "supervisor", "head", "staff"]
PROV_KEEP = {"Jakarta Raya", "Banten", "Jawa Barat", "Jawa Timur", "Bali"}
WORK_TYPE_MAP = {
    "Full time": "Full-time",
    "Kontrak/Temporer": "Contract",
    "Paruh waktu": "Part-time",
    "Kasual": "Casual",
}
SENIORITY_LABEL = {
    "intern": "Intern", "junior": "Junior", "staff": "Staff", "supervisor": "Supervisor",
    "senior": "Senior", "lead": "Lead", "principal": "Lead", "manager": "Manager", "head": "Head",
}
NS_JOBS = uuid.UUID("6f9b4d1e-0000-4000-8000-000000000001")  # namespace tetap utk UUID5


def parse_salary(s):
    t = str(s).strip()
    if t.lower() in {"none", "null", "nan", "", "-"}:
        return pd.Series({"salary_min": np.nan, "salary_max": np.nan, "period": None})
    low = t.lower()
    period = "month" if ("month" in low or "bulan" in low) else (
        "year" if ("year" in low or "annum" in low or "tahun" in low) else (
        "hour" if ("hour" in low or "jam" in low) else None))
    cleaned = t.replace("\xa0", " ").replace(".", "").replace(",", "")
    nums = [int(n) for n in re.findall(r"\d+", cleaned) if len(n) >= 4]
    if len(nums) >= 2:
        return pd.Series({"salary_min": min(nums[:2]), "salary_max": max(nums[:2]), "period": period})
    if len(nums) == 1:
        return pd.Series({"salary_min": nums[0], "salary_max": nums[0], "period": period})
    return pd.Series({"salary_min": np.nan, "salary_max": np.nan, "period": period})


def parse_location(loc):
    s = str(loc)
    low = s.lower()
    arrangement = "Hybrid" if ("hibrid" in low or "hybrid" in low) else (
        "Remote" if ("remote" in low or "jarak jauh" in low) else "Onsite")
    s2 = re.sub(r"\(.*?\)", "", s).replace("\n", " ").strip()
    parts = [p.strip() for p in s2.split(",") if p.strip()]
    city = parts[0] if len(parts) >= 2 else None
    province = parts[-1] if parts else "Unknown"
    return city, province, arrangement


def extract_years(text_):
    m = re.search(r"(\d+)\s*\+?\s*(?:years?|tahun|thn)", str(text_), re.I)
    return float(m.group(1)) if m else np.nan


def seniority_keyword(title):
    t = str(title).lower()
    for kw in SENIORITY_KEYWORDS:
        if kw in t:
            return kw
    return "unspecified"


def role_family(title):
    t = str(title).lower()
    if any(k in t for k in ["data", "analyst", "scientist", "machine learning"]):
        return "Data/Analytics"
    if any(k in t for k in ["engineer", "developer", "programmer", "software", "devops", "qa"]):
        return "Engineering/IT"
    if any(k in t for k in ["sales", "marketing", "business development"]):
        return "Sales/Marketing"
    if any(k in t for k in ["hr", "human resource", "recruit", "talent", "people"]):
        return "HR"
    if any(k in t for k in ["design", "creative", "content", "graphic", "ui", "ux"]):
        return "Design/Creative"
    if any(k in t for k in ["finance", "accounting", "accountant", "tax", "audit"]):
        return "Finance"
    if any(k in t for k in ["manager", "supervisor", "head", "director"]):
        return "Management"
    return "Other"


def build_features(frame):
    """Fitur untuk model gaji — identik dgn salary_ml.ipynb."""
    out = []
    for _, r in frame.iterrows():
        _, province, arrangement = parse_location(r["location"])
        province = province if province in PROV_KEEP else "Other"
        out.append({
            "years_exp": extract_years(r["job_description"]),
            "seniority": seniority_keyword(r["job_title"]),
            "role": role_family(r["job_title"]),
            "province": province,
            "work_type": r["work_type"],
            "arrangement": arrangement,
            "desc_wordcount": len(str(r["job_description"]).split()),
        })
    return pd.DataFrame(out, index=frame.index)


def seniority_level(title, years):
    kw = seniority_keyword(title)
    if kw != "unspecified":
        return SENIORITY_LABEL.get(kw)
    if pd.notna(years):
        if years < 2:
            return "Entry"
        if years < 5:
            return "Mid"
        return "Senior"
    return None


def make_job_id(row):
    key = f"{row['job_title']}|{row['company_name']}|{row['location']}|{row['job_description']}"
    return str(uuid.uuid5(NS_JOBS, key))


print("Fungsi transformasi siap.")

Fungsi transformasi siap.


In [4]:
# --- Muat, dedup, transformasi ---
DATA_PATH = resolve_path("dataset/jobs.jsonl")
records = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))
raw = pd.DataFrame(records)
raw = raw.drop_duplicates(
    subset=["job_title", "job_description", "company_name", "location"]
).reset_index(drop=True)

sal = raw["salary"].apply(parse_salary)
loc = raw["location"].apply(lambda x: pd.Series(parse_location(x), index=["city", "province", "work_arrangement"]))
years = raw["job_description"].apply(extract_years)

df = pd.DataFrame({
    "job_title": raw["job_title"],
    "company_name": raw["company_name"].where(
        ~raw["company_name"].str.contains("anonim|anonymous|confidential|rahasia", case=False, regex=True), None),
    "city": loc["city"],
    "province": loc["province"],
    "work_type": raw["work_type"].map(WORK_TYPE_MAP).fillna(raw["work_type"]),
    "work_arrangement": loc["work_arrangement"],
    "seniority_level": [seniority_level(t, y) for t, y in zip(raw["job_title"], years)],
    "salary_min": sal["salary_min"],
    "salary_max": sal["salary_max"],
    "salary_period": sal["period"],
    "job_description": raw["job_description"],
    "location": raw["location"],  # dipakai build_features utk prediksi gaji
    "created_at": pd.to_datetime(raw["_scrape_timestamp"]).dt.strftime("%Y-%m-%d %H:%M:%S"),
})
df["job_id"] = raw.apply(make_job_id, axis=1)
print(f"Baris siap proses: {len(df)}")
df.head(3)

Baris siap proses: 465


,job_title,company_name,city,province,work_type,work_arrangement,seniority_level,salary_min,salary_max,salary_period,job_description,location,created_at,job_id
0,Data Analyst,PT Matahari Department Store Tbk,Karawaci,Banten,Full-time,Onsite,Mid,NaN,NaN,NaN,"Responsibilities:\nCollect, manage, and analyz...","Karawaci, Banten",2025-11-24 11:33:16,858c1af8-d318-5122-85bb-c0e6654d020c
1,Data Analyst Intern,"PT Surya Semesta Internusa, Tbk",Jakarta Selatan,Jakarta Raya,Part-time,Onsite,Intern,NaN,NaN,NaN,Job Description\nSupport in conducting thoroug...,"Jakarta Selatan, Jakarta Raya",2025-11-24 11:33:32,07190c9a-126c-5073-8283-0240924bc90e
2,Data Analyst Supervisor,PT Pangan Lestari,Surabaya,Jawa Timur,Full-time,Onsite,Supervisor,NaN,NaN,NaN,Tugas dan Tanggung Jawab:\nMembuat dan mengemb...,"Surabaya, Jawa Timur",2025-11-24 11:33:48,71dc28fb-4c12-5e50-850a-eabb32120b71


In [5]:
# --- Isi gaji kosong dengan model salary_ml + tandai is_salary_estimated ---
model = joblib.load(resolve_path("models/salary_model.joblib"))

need = df["salary_min"].isna()
df["is_salary_estimated"] = need
if need.any():
    feats = build_features(df[need])
    preds = model.predict(feats)  # juta Rupiah
    est = (np.round(preds * 1_000_000, -4)).astype(int)
    df.loc[need, "salary_min"] = est
    df.loc[need, "salary_max"] = est
    df.loc[need, "salary_period"] = "month"

df["salary_min"] = df["salary_min"].astype(int)
df["salary_max"] = df["salary_max"].astype(int)
print(f"Gaji asli     : {(~df['is_salary_estimated']).sum()}")
print(f"Gaji estimasi : {df['is_salary_estimated'].sum()}")
df[["job_title", "salary_min", "salary_max", "is_salary_estimated"]].head()

Gaji asli     : 177
Gaji estimasi : 288


,job_title,salary_min,salary_max,is_salary_estimated
0,Data Analyst,8470000,8470000,True
1,Data Analyst Intern,12550000,12550000,True
2,Data Analyst Supervisor,10450000,10450000,True
3,Data Analyst,9580000,9580000,True
4,Data Analyst,7240000,7240000,True


## Bagian 3 — Ingest ke MySQL (idempotent)

Buat tabel `jobs` bila belum ada, lalu `INSERT ... ON DUPLICATE KEY UPDATE` dengan `job_id` sebagai PK. Menjalankan ulang hanya memperbarui baris yang sama, tidak menggandakan.

In [6]:
CREATE_SQL = """
CREATE TABLE IF NOT EXISTS jobs (
    job_id           CHAR(36)     PRIMARY KEY,
    job_title        VARCHAR(255) NOT NULL,
    company_name     VARCHAR(255) NULL,
    city             VARCHAR(120) NULL,
    province         VARCHAR(120) NULL,
    work_type        VARCHAR(50)  NULL,
    work_arrangement VARCHAR(20)  NULL,
    seniority_level  VARCHAR(30)  NULL,
    salary_min       INT          NULL,
    salary_max       INT          NULL,
    salary_period    VARCHAR(20)  NULL,
    is_salary_estimated TINYINT(1) NOT NULL DEFAULT 0,
    job_description  TEXT         NULL,
    created_at       DATETIME     NULL
);
"""

COLS = ["job_id", "job_title", "company_name", "city", "province", "work_type",
        "work_arrangement", "seniority_level", "salary_min", "salary_max",
        "salary_period", "is_salary_estimated", "job_description", "created_at"]

UPSERT_SQL = (
    "INSERT INTO jobs (" + ", ".join(COLS) + ") VALUES (" + ", ".join(f":{c}" for c in COLS) + ") "
    "ON DUPLICATE KEY UPDATE " + ", ".join(f"{c}=VALUES({c})" for c in COLS if c != "job_id")
)


def to_record(row):
    rec = {}
    for c in COLS:
        v = row[c]
        if c == "is_salary_estimated":
            rec[c] = int(bool(v))
        elif pd.isna(v):
            rec[c] = None
        elif isinstance(v, (np.integer,)):
            rec[c] = int(v)
        else:
            rec[c] = v
    return rec


records_sql = [to_record(r) for _, r in df.iterrows()]
with engine.begin() as conn:
    conn.execute(text(CREATE_SQL))
    conn.execute(text(UPSERT_SQL), records_sql)

with engine.connect() as conn:
    total = conn.execute(text("SELECT COUNT(*) FROM jobs")).scalar()
print(f"Upsert selesai. Total baris di tabel jobs: {total}")

Upsert selesai. Total baris di tabel jobs: 465


Buat table `learning` bila belum ada, untuk mencari situs yang tepat untuk belajar. Lalu `INSERT ... ON DUPLICATE KEY UPDATE` dengan `id` sebagai PK. Menjalankan ulang hanya memperbarui baris yang sama, tidak menggandakan.

In [13]:
from sqlalchemy import text
CREATE_SQL = """CREATE TABLE IF NOT EXISTS learning_links (
    id INT AUTO_INCREMENT PRIMARY KEY,
    category VARCHAR(100) NOT NULL,
    title VARCHAR(255) NOT NULL,
    url TEXT NOT NULL,
    provider VARCHAR(100),
    free BOOLEAN
);"""

LEARNING_LINKS = {"sql": [
        ("Dicoding — Belajar Dasar SQL",
         "https://www.dicoding.com/academies/191",    "Dicoding 🇮🇩", True),
        ("Coursera — SQL for Data Science",
         "https://www.coursera.org/learn/sql-for-data-science", "Coursera", True),
        ("W3Schools SQL Tutorial",
         "https://www.w3schools.com/sql/",            "W3Schools",    True),
        ("HackerRank SQL Practice",
         "https://www.hackerrank.com/domains/sql",    "HackerRank",   True),
    ],
    "python": [
        ("Dicoding — Memulai Python",
         "https://www.dicoding.com/academies/86",     "Dicoding 🇮🇩", True),
        ("Coursera — Python for Everybody",
         "https://www.coursera.org/specializations/python", "Coursera", True),
        ("Kaggle — Python (interaktif)",
         "https://www.kaggle.com/learn/python",       "Kaggle",       True),
    ],
    "excel": [
        ("Microsoft Learn — Excel",
         "https://support.microsoft.com/en-us/excel", "Microsoft",    True),
        ("Coursera — Excel Skills for Business",
         "https://www.coursera.org/specializations/excel", "Coursera", True),
        ("GCFGlobal — Excel Tutorial",
         "https://edu.gcfglobal.org/en/excel2016/",   "GCFGlobal",    True),
    ],
    "tableau": [
        ("Tableau eLearning (official)",
         "https://www.tableau.com/learn/training/elearning", "Tableau", True),
        ("Kaggle — Data Visualization",
         "https://www.kaggle.com/learn/data-visualization", "Kaggle", True),
    ],
    "power bi": [
        ("Microsoft Learn — Power BI",
         "https://learn.microsoft.com/en-us/power-bi/", "Microsoft",  True),
        ("Coursera — Power BI Data Analyst",
         "https://www.coursera.org/professional-certificates/"
         "microsoft-power-bi-data-analyst",           "Coursera",     False),
    ],
    "marketing": [
        ("Google Digital Marketing",
         "https://grow.google/intl/id_id/",           "Google 🇮🇩",   True),
        ("Meta Blueprint",
         "https://www.facebook.com/business/learn",   "Meta",         True),
    ],
    "analytics": [
        ("Google Analytics — Skillshop",
         "https://skillshop.withgoogle.com/",         "Google",       True),
        ("Coursera — Google Data Analytics",
         "https://www.coursera.org/professional-certificates/"
         "google-data-analytics",                     "Coursera",     False),
        ("Kaggle — Pandas",
         "https://www.kaggle.com/learn/pandas",       "Kaggle",       True),
    ],
    "communication": [
        ("Coursera — Business Communication",
         "https://www.coursera.org/specializations/business-communication",
         "Coursera", True),
    ],
    "adobe": [
        ("Adobe Learn — Tutorial Resmi",
         "https://helpx.adobe.com/id/learn-and-support.html", "Adobe", True),
    ],
    "design": [
        ("Dicoding — Belajar Dasar UI/UX",
         "https://www.dicoding.com/academies/428",    "Dicoding 🇮🇩", True),
        ("Coursera — Google UX Design",
         "https://www.coursera.org/professional-certificates/google-ux-design",
         "Coursera", False),
        ("Figma Learn",
         "https://help.figma.com/hc/en-us/categories/360002051613", "Figma", True),
    ],
    "statistics": [
        ("Coursera — Statistics with Python",
         "https://www.coursera.org/specializations/statistics-with-python",
         "Coursera", True),
        ("Khan Academy — Statistics",
         "https://www.khanacademy.org/math/statistics-probability",
         "Khan Academy", True),
    ],
}

UPSERT_SQL = """
INSERT INTO learning_links
(category, title, url, provider, free)
VALUES
(:category, :title, :url, :provider, :free)
ON DUPLICATE KEY UPDATE
url = VALUES(url),
provider = VALUES(provider),
free = VALUES(free)
"""

records = []

for category, links in LEARNING_LINKS.items():
    for title, url, provider, free in links:
        records.append({
            "category": category,
            "title": title,
            "url": url,
            "provider": provider,
            "free": free,
        })

with engine.begin() as conn:
    conn.execute(text(CREATE_SQL))
    conn.execute(text(UPSERT_SQL), records)

with engine.connect() as conn:
    total = conn.execute(
        text("SELECT COUNT(*) FROM learning_links")).scalar()
print(f"Inserted {total} rows.")

Inserted 26 rows.


## Bagian 4 — Ingest ke Qdrant (idempotent + resume)

Teks yang di-embed = `job_title` + `job_description`, dan **teks yang sama disimpan di payload `content`** agar retriever LangChain (node Qdrant di n8n) membaca isi dokumen yang identik dengan dasar pencarian vektor. Field terstruktur disimpan di `metadata` (dan di-flatten di level atas) untuk filter hybrid. `point_id = job_id`. Sebelum embedding, `job_id` yang sudah ada di collection diambil dan dilewati (mekanisme resume).

In [ ]:
from qdrant_client.models import PointStruct, VectorParams, Distance

# Buat collection bila belum ada
if not qc.collection_exists(QDRANT_COLLECTION):
    qc.create_collection(
        QDRANT_COLLECTION,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    )
    print(f"Collection '{QDRANT_COLLECTION}' dibuat.")
else:
    print(f"Collection '{QDRANT_COLLECTION}' sudah ada.")


def existing_point_ids():
    """Ambil semua job_id yang sudah tersimpan di Qdrant (untuk resume)."""
    ids, offset = set(), None
    while True:
        pts, offset = qc.scroll(QDRANT_COLLECTION, with_payload=False,
                                with_vectors=False, limit=256, offset=offset)
        ids.update(str(p.id) for p in pts)
        if offset is None:
            break
    return ids


def embed_texts(texts):
    resp = client_oai.embeddings.create(model=EMB_MODEL, input=texts)
    return [d.embedding for d in resp.data]


def build_content(row):
    """Teks dokumen yang di-embed sekaligus disimpan ke payload.

    Menjadi satu-satunya sumber teks: dipakai sebagai input embedding pada
    loop upsert DAN disimpan di payload `content`. Dengan begitu isi yang
    dibaca retriever LangChain (node Qdrant di n8n) identik dengan teks yang
    menjadi dasar pencarian vektor.
    """
    return f"{row['job_title']}\n{row['job_description']}"


def make_payload(row):
    """Payload Qdrant untuk satu lowongan.

    Struktur mengikuti format node Qdrant Vector Store di n8n:
        - ``content``  : teks dokumen (pageContent) yang dibaca agent RAG.
        - ``metadata`` : field terstruktur untuk ditampilkan/difilter.
    Field terstruktur juga di-flatten di level atas agar bisa dipakai untuk
    filter langsung di Qdrant tanpa menembus objek metadata.
    """
    meta = {
        "job_id": row["job_id"],
        "job_title": row["job_title"],
        "company_name": row["company_name"] if pd.notna(row["company_name"]) else None,
        "city": row["city"] if pd.notna(row["city"]) else None,
        "province": row["province"],
        "work_type": row["work_type"],
        "work_arrangement": row["work_arrangement"],
        "seniority_level": row["seniority_level"] if pd.notna(row["seniority_level"]) else None,
        "salary_min": int(row["salary_min"]),
        "salary_max": int(row["salary_max"]),
        "is_salary_estimated": bool(row["is_salary_estimated"]),
    }
    return {**meta, "content": build_content(row), "metadata": meta}


print("Helper Qdrant siap.")

In [ ]:
# --- Loop embedding + upsert per batch (resume-aware) ---
done = existing_point_ids()
todo = df[~df["job_id"].isin(done)].reset_index(drop=True)
print(f"Sudah ada di Qdrant : {len(done)}")
print(f"Perlu diproses      : {len(todo)}")

n_upserted = 0
for start in range(0, len(todo), BATCH_SIZE):
    batch = todo.iloc[start:start + BATCH_SIZE]
    # Teks embedding = payload `content` (via build_content) → satu sumber teks.
    texts = [build_content(r) for _, r in batch.iterrows()]
    vectors = embed_texts(texts)
    points = [
        PointStruct(id=r["job_id"], vector=vec, payload=make_payload(r))
        for (_, r), vec in zip(batch.iterrows(), vectors)
    ]
    qc.upsert(collection_name=QDRANT_COLLECTION, points=points)
    n_upserted += len(points)
    print(f"  batch {start // BATCH_SIZE + 1}: +{len(points)} (total {n_upserted}/{len(todo)})")

count = qc.count(QDRANT_COLLECTION).count
print(f"\nSelesai. Total titik di collection '{QDRANT_COLLECTION}': {count}")

## Bagian 5 — Verifikasi

Cek konsistensi jumlah MySQL vs Qdrant, uji idempotency (hitung ulang yang perlu diproses — harus 0), dan lakukan satu pencarian semantik contoh.

In [9]:
with engine.connect() as conn:
    mysql_count = conn.execute(text("SELECT COUNT(*) FROM jobs")).scalar()
qdrant_count = qc.count(QDRANT_COLLECTION).count
remaining = df[~df["job_id"].isin(existing_point_ids())]
print(f"MySQL rows          : {mysql_count}")
print(f"Qdrant points       : {qdrant_count}")
print(f"Sisa belum di Qdrant: {len(remaining)}  (idempotent bila 0)")

MySQL rows          : 465
Qdrant points       : 465
Sisa belum di Qdrant: 0  (idempotent bila 0)


In [10]:
# Pencarian semantik contoh (RAG end-to-end)
query = "data analyst berpengalaman di Jakarta"
qvec = embed_texts([query])[0]
hits = qc.query_points(QDRANT_COLLECTION, query=qvec, limit=5, with_payload=True).points
print(f"Query: {query!r}\n")
for h in hits:
    p = h.payload
    print(f"  {h.score:.3f} | {p['job_title'][:45]:45s} | {p.get('city') or p['province']:20s} | "
          f"Rp {p['salary_min']:,}{' (est)' if p['is_salary_estimated'] else ''}")

Query: 'data analyst berpengalaman di Jakarta'

  0.737 | Data Analyst                                  | Jakarta Raya         | Rp 6,000,000
  0.713 | Data Analyst                                  | Jakarta Selatan      | Rp 6,750,000
  0.679 | Data Analyst                                  | Cirebon              | Rp 10,000,000
  0.671 | Data Analyst                                  | Bogor                | Rp 7,240,000 (est)
  0.667 | Data Analyst                                  | Tangerang District   | Rp 8,450,000 (est)


## Bagian 6 — Kesimpulan

**Hasil ingestion:**
- **MySQL** (`jobs`): 465 baris (setelah dedup 8 duplikat dari 473).
- **Qdrant** (`jobs`): 465 titik vektor (1536-dim, cosine), `point_id = job_id`.
- **Gaji**: 177 asli, 288 estimasi model (`is_salary_estimated = TRUE`).
- **Idempotency terverifikasi**: sisa yang belum masuk Qdrant = 0; menjalankan ulang tidak menggandakan (MySQL `ON DUPLICATE KEY UPDATE`, Qdrant `upsert` id sama).
- **Pencarian semantik** berfungsi end-to-end (contoh query mengembalikan lowongan Data Analyst yang relevan beserta gaji & penanda estimasi).

**Mekanisme yang bekerja:**
- `job_id` UUID5 deterministik → jembatan MySQL↔Qdrant + dasar anti-duplikat.
- **Resume**: batch berikutnya melewati `job_id` yang sudah ada di Qdrant; kegagalan di tengah dapat dilanjutkan tanpa membuang biaya embedding.

Detail skema: [`docs/erd.md`](../docs/erd.md) (MySQL) & [`docs/qdrant.md`](../docs/qdrant.md).